# When Can You Trust an IQB Score?

A country's IQB score is calculated from however many speed tests happened to be run there that
month — 36 million in India, one in Pitcairn. This notebook explains how we work out when that is
enough, and why the answer is completely different for every country.

**The headline criterion.** For planning, we use the smallest sample size for which the probability
of being within +/-0.05 IQB units of the target score is at least 95%. The target in the main
analysis is the country's 24-month pooled distribution at p50. This is an iid model-based lower
bound, not a guarantee for a single current month: measurements are clustered and the country’s
underlying distribution changes over time.

An exploratory diagnostic reports an expected sampling-error crossover near 100 measurements for
a typical country. That diagnostic is useful for comparing sampling noise with month-to-month
movement, but it is not a 95% confidence requirement.

It is written to be read rather than run. Every number in it is computed from the data below, so
nothing is quoted from memory, but the argument is the point and the code is there to back it up.

**Where the other notebooks fit.** `03-sample_size_reliability.ipynb` is the primary calculation;
`04-mad_vs_sample_size.ipynb` measures observed month-to-month movement;
`05-methodology_review.ipynb` audits the assumptions; and `06-evaluating_mad.ipynb` explains the
role and limits of MAD. `METHODOLOGY.md` is the written method. This notebook is the explanation.

---

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycountry
import pyarrow.parquet as pq
from scipy import stats

from iqb import IQBCalculator
from iqb.calculator.config import IQB_CONFIG

calculator = IQBCalculator()

DATA_DIR = Path("./data/v1")
FIG_DIR = Path("./analysis_outputs/explainer_figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

PERCENTILES = [1, 5, 10, 25, 50, 75, 90, 95, 99]
QLEV = np.array(PERCENTILES) / 100.0
METRICS = ["download", "upload", "latency", "loss"]
RAW_KEY = {"download": "download_throughput_mbps", "upload": "upload_throughput_mbps",
           "latency": "latency_ms", "loss": "packet_loss"}
NICE = {"download_throughput_mbps": "download", "upload_throughput_mbps": "upload",
        "latency_ms": "latency", "packet_loss": "loss"}
HIGHER_BETTER = {"download": True, "upload": True, "latency": False, "loss": False}
UNIT = {"download": "Mbps", "upload": "Mbps", "latency": "ms", "loss": ""}
TINY = 1e-9

INK, ACCENT, SLATE, WARN, GOOD = "#14171A", "#9A6410", "#43596B", "#A63A2E", "#3C6E4C"
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.22,
                     "grid.linestyle": "--", "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})


def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight", facecolor="white")


def country_name(cc):
    c = pycountry.countries.get(alpha_2=cc)
    return c.name if c else cc


def load_all(root: Path = DATA_DIR) -> pd.DataFrame:
    frames = []
    for pdir in sorted(p for p in root.iterdir() if p.is_dir()):
        for edir in sorted(pdir.iterdir()):
            dl, ul = (edir / "downloads_by_country" / "data.parquet",
                      edir / "uploads_by_country" / "data.parquet")
            if not (dl.exists() and ul.exists()):
                continue
            if dl.stat().st_size < 1000 or ul.stat().st_size < 1000:
                continue
            a, b = pq.read_table(dl).to_pandas(), pq.read_table(ul).to_pandas()
            m = a.merge(b, on="country_code", suffixes=("_down", "_up"))
            m["n_down"] = m["sample_count_down"].astype(float)
            m["n_up"] = m["sample_count_up"].astype(float)
            m["month"] = f"{pdir.name[:4]}-{pdir.name[4:6]}"
            frames.append(m)
    return pd.concat(frames, ignore_index=True)


df = load_all()
latest = df[df.month == df.month.max()].reset_index(drop=True)
print(f"{len(df):,} country-months  ({df.country_code.nunique()} countries x "
      f"{df.month.nunique()} months, {df.month.min()} .. {df.month.max()})")

---

## 2. The Score Is Ten Yes/No Questions

This is the claim everything else rests on. It is not obvious from the configuration file, so this
section goes through it one step at a time. The config looks like a tree of weighted averages. The claim is that
the whole tree collapses into a single weighted sum of ten yes/no answers.

### 2a. What the calculator actually does

`iqb.calculator` computes the score in **three nested weighted averages**, innermost first:

| level | function | what it averages |
|---|---|---|
| 3 | `_calculate_binary_requirement_score` | nothing — it returns **1 or 0** |
| 3 | `_calculate_requirement_agreement_score` | those 1s and 0s, weighted **across datasets** |
| 2 | `_calculate_use_case_score` | requirement scores, weighted **across the 4 requirements** |
| 1 | `_calculate_iqb_score` | use-case scores, weighted **across the 6 use cases** |

The bottom level is where the "yes/no" comes from, and it is completely explicit in the source:

```python
def _calculate_binary_requirement_score(*, network_requirement, value, threshold) -> int:
    if network_requirement == "download_throughput_mbps":
        return 1 if value > threshold else 0
    ...
    elif network_requirement == "packet_loss":
        return 1 if value < threshold else 0
```

There is no partial credit anywhere. A country either clears 25 Mbps or it does not.

**The dataset level disappears.** The requirement-agreement score averages the binary score across
data sources, but in the default config only `m-lab` carries a non-zero weight — `cloudflare` and
`ookla` are both 0. A weighted average with one non-zero weight is just that value, so this level
contributes nothing and the requirement score stays exactly 1 or 0.

In [ ]:
ds = {}
for uc in IQB_CONFIG["use cases"].values():
    for nr in uc["network requirements"].values():
        for name, cfg in nr["datasets"].items():
            ds[name] = ds.get(name, 0) + cfg["w"]
print("total weight each data source carries across the whole config:")
for k, v in ds.items():
    print(f"   {k:12s}{v}")
print("\n-> only m-lab contributes, so the dataset average returns the binary score unchanged.")

### 2b. Two nested weighted averages are one weighted sum

That leaves two levels. Write $W = \sum_u w_u$ for the total use-case weight and
$W_u = \sum_r w_r$ for the total requirement weight inside use case $u$. Then

$$\text{use case score}_u \;=\; \sum_r \frac{w_r}{W_u}\,\mathbf{1}\{\text{test}_{u,r}\}
\qquad\text{and}\qquad
\mathrm{IQB} \;=\; \sum_u \frac{w_u}{W}\;\text{use case score}_u$$

Substituting the first into the second and multiplying the weights through:

$$\mathrm{IQB} \;=\; \sum_u \sum_r
\underbrace{\frac{w_u}{W}\cdot\frac{w_r}{W_u}}_{a_{u,r}}\;\mathbf{1}\{\text{test}_{u,r}\}$$

**That is the whole trick.** A weighted average of weighted averages is a weighted sum with the
weights multiplied together. There is no approximation in that step — it is just expanding brackets.

The weights still add to 1, because each inner set adds to 1 on its own:

$$\sum_u \sum_r a_{u,r} \;=\; \sum_u \frac{w_u}{W} \underbrace{\sum_r \frac{w_r}{W_u}}_{=\,1}
\;=\; \sum_u \frac{w_u}{W} \;=\; 1$$

So the score is a weighted sum over **24 leaves** — 6 use cases × 4 requirements — each leaf a
yes/no answer carrying a fixed share of the scale.

In [ ]:
W = sum(uc["w"] for uc in IQB_CONFIG["use cases"].values())
leaves = []
for uc_name, uc in IQB_CONFIG["use cases"].items():
    Wu = sum(nr["w"] for nr in uc["network requirements"].values())
    for nr_name, nr in uc["network requirements"].items():
        m = NICE[nr_name]
        leaves.append({
            "use case": uc_name, "metric": m,
            "uc share": uc["w"] / W, "req share": nr["w"] / Wu,
            "a": (uc["w"] / W) * (nr["w"] / Wu),
            "question": f"{m} {'>' if HIGHER_BETTER[m] else '<'} {nr['threshold min']:g}"
                        f"{(' ' + UNIT[m]) if UNIT[m] else ''}",
        })
L = pd.DataFrame(leaves)

print(f"the {len(L)} leaves, with the two weights multiplied through\n")
print(f"{'use case':20s}{'yes/no question':26s}{'use case share':>16s}"
      f"{'req share':>11s}{'leaf weight':>13s}")
for uc_name, g in L.groupby("use case", sort=False):
    for j, (_, r) in enumerate(g.iterrows()):
        print(f"{(uc_name if j == 0 else ''):20s}{r['question']:26s}"
              f"{r['uc share']:>16.4f}{r['req share']:>11.4f}{r['a']:>13.6f}")
print(f"{'':57s}{'TOTAL':>16s}{L.a.sum():>13.6f}")

### 2c. Several leaves ask the same question, so they merge

Look down the question column above and you will see repeats. `download > 10 Mbps` appears in web
browsing, audio streaming and online backup. Those are three separate leaves in the config, but
they are the *same yes/no question about the same number* — if a country's download clears 10 Mbps
for one of them, it clears it for all three.

So the indicator can be factored out and the three weights added:

$$\mathrm{IQB} \;=\; \sum_{k=1}^{10}
\Bigl(\underbrace{\textstyle\sum_{(u,r)\,\to\,k} a_{u,r}}_{w_k}\Bigr)\,
\mathbf{1}\{\text{question}_k\}
\;=\; \sum_{k=1}^{10} w_k\,\mathbf{1}\{\text{question}_k\}$$

24 leaves, but only **10 distinct questions**. That is the whole reduction.

In [ ]:
T = (L.groupby("question", as_index=False)
       .agg(weight=("a", "sum"), leaves=("a", "size"),
            from_=("use case", lambda s: ", ".join(sorted(s))))
       .sort_values("weight", ascending=False).reset_index(drop=True))

print(f"{'the ten questions':26s}{'weight':>9s}{'leaves':>8s}  built from")
for _, r in T.iterrows():
    print(f"{r['question']:26s}{r.weight:>9.4f}{r.leaves:>8d}  {r.from_}")
print(f"{'TOTAL':26s}{T.weight.sum():>9.4f}{T.leaves.sum():>8d}")

### 2d. Seeing the collapse

The matrix below is the same fact drawn out. Six rows, one per use case. Ten columns, one per
question. A filled cell is a leaf, and its value is that leaf's weight. Each row adds to that use
case's share of the score. **Each column total is one of the ten weights** — the thing the whole
reduction produces.

In [ ]:
order = T.question.tolist()
ucs = list(IQB_CONFIG["use cases"].keys())
M = np.zeros((len(ucs), len(order)))
for _, r in L.iterrows():
    M[ucs.index(r["use case"]), order.index(r["question"])] += r["a"]

fig, ax = plt.subplots(figsize=(12.4, 3.9))
ax.imshow(np.where(M > 0, M, np.nan), aspect="auto", cmap="YlOrBr", vmin=0, vmax=M.max() * 1.5)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        if M[i, j] > 0:
            ax.text(j, i, f"{M[i, j]:.3f}", ha="center", va="center", fontsize=8.5, color=INK)
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order, rotation=32, ha="right", fontsize=8.5)
ax.set_yticks(range(len(ucs)))
ax.set_yticklabels(ucs, fontsize=9)
ax.set_xticks(np.arange(-.5, len(order), 1), minor=True)
ax.set_yticks(np.arange(-.5, len(ucs), 1), minor=True)
ax.grid(which="minor", color="white", lw=1.6)
ax.grid(which="major", visible=False)
ax.tick_params(which="minor", length=0)
ax.set_ylim(len(ucs) - 0.5, -1.6)          # room for a totals row above the matrix
for j in range(len(order)):
    ax.text(j, -1.05, f"{M[:, j].sum():.3f}", ha="center", va="center",
            fontsize=9.5, fontweight="bold", color=ACCENT)
ax.text(-0.7, -1.05, "the ten weights", ha="right", va="center", fontsize=9,
        fontweight="bold", color=ACCENT)
ax.set_title("24 leaves land in 10 columns. Each column total is one of the ten weights.",
             fontsize=11, pad=16)
save(fig, "fig1_the_collapse")
plt.show()

print("row totals (each use case's share of the score):",
      "  ".join(f"{v:.4f}" for v in M.sum(axis=1)))
print(f"they are all 1/6 = {1/6:.4f}, because the six use cases carry equal weight in the config.")

### 2e. Checking it against the real calculator

The derivation above is algebra, so it should reproduce `IQBCalculator` exactly — not
approximately. The cell below scores 400 real country-months at all nine percentile levels both
ways and reports the largest disagreement. If that number is not at machine precision, the
reduction is wrong and nothing later in this notebook holds.

In [ ]:
ATOMS = {m: [] for m in METRICS}
for _, r in T.iterrows():
    parts = r["question"].split()
    ATOMS[parts[0]].append((float(parts[2]), r.weight))
for m in ATOMS:
    ATOMS[m].sort(key=lambda t: t[0], reverse=not HIGHER_BETTER[m])
ATOM_LABELS = [f"{m} {'>' if HIGHER_BETTER[m] else '<'} {t:g}" for m in METRICS for t, _ in ATOMS[m]]
ATOM_W = np.array([w for m in METRICS for _, w in ATOMS[m]])


def iqb_ten_questions(vals: dict) -> float:
    """The whole IQB score, as ten weighted yes/no answers."""
    return sum(w * ((vals[m] > thr) if HIGHER_BETTER[m] else (vals[m] < thr))
               for m in METRICS for thr, w in ATOMS[m])


check = df.sample(400, random_state=0)
worst = max(
    abs(iqb_ten_questions({m: float(r[f"{m}_p{p}"]) for m in METRICS})
        - calculator.calculate_iqb_score(
            data={"m-lab": {RAW_KEY[m]: float(r[f"{m}_p{p}"]) for m in METRICS}}))
    for _, r in check.iterrows() for p in PERCENTILES)
print(f"largest disagreement over {len(check)} country-months x 9 levels: {worst:.2e}")
print("(machine precision is about 2e-16, so the two are the same calculation)")

### What this buys us

Because the score is a weighted sum of ten yes/no answers:

> **The score cannot be slightly wrong. If it is wrong, one of the ten answers came out on the
> wrong side — and the score is then wrong by that answer's entire weight.**

A published score of 0.68 when the truth is 0.88 does not mean the calculation was a bit off. It
means the packet-loss question was answered "no" when the honest answer was "yes".

That turns a hard question — *how many measurements before a score is accurate?* — into a much
easier one: *how many measurements before ten yes/no answers are right?* The rest of this notebook
is about the second question.

Note also how uneven the weights are. Six of the ten questions weigh more than 0.05 on their own,
so any one of them coming out wrong breaks a ±0.05 tolerance by itself.

---

## 3. Nine Scores, Not One

The pipeline does not publish one score per country. It publishes nine, one for each percentile
level. **p50** describes the typical user in a country, **p90** a well-served one, **p10** someone
poorly served.

All nine come from exactly the same measurements. What changes is the question asked of them — and
written out, each level turns into a **share**.

If $\pi$ is the fraction of a country's measurements that pass a given threshold, then the
$q$-quantile passes that threshold exactly when $q > 1 - \pi$. Which is to say:

In [ ]:
print(f"{'scoring at':>12s}   is really asking")
for p in PERCENTILES:
    print(f"{'p' + str(p):>12s}   do more than {100 - p}% of measurements pass?")
print("\nSo the nine levels are nine different cut-offs applied to the SAME count of passing")
print("measurements. That is the idea the next section makes concrete.")

### Reading a threshold off the nine percentiles

Two helpers used from here on. `to_quality` puts every metric on a scale where higher is better, so one rule covers all four. `pass_prob` reads the share of measurements that pass a threshold off a country's nine reported percentiles — it is the step everything else depends on.


In [ ]:
def to_quality(metric, v):
    """Map raw values into a space where higher is always better and passing means y > t."""
    v = np.asarray(v, dtype=float)
    if metric in ("download", "upload"):
        return np.log(np.maximum(v, TINY))      # log: throughput spans decades
    if metric == "latency":
        return -np.log(np.maximum(v, TINY))
    return -v                                    # loss: linear, it has a point mass at 0


def pass_prob(y_knots, t):
    """What share of measurements pass, read off the nine reported percentiles."""
    y = np.asarray(y_knots, float)
    uy = np.unique(y)
    q_lo = np.array([QLEV[y == v].min() for v in uy])
    q_hi = np.array([QLEV[y == v].max() for v in uy])
    if t <= uy[0]:
        return 0.995
    if t >= uy[-1]:
        return 0.005
    i = np.searchsorted(uy, t) - 1
    F = q_hi[i] + (q_lo[i + 1] - q_hi[i]) * (t - uy[i]) / (uy[i + 1] - uy[i])
    return float(np.clip(1.0 - F, 0.005, 0.995))


# Upload percentiles come from the uploads table, the other three from the downloads table,
# so the two directions carry different measurement counts.
BACKED_BY = {"download": "down", "upload": "up", "latency": "down", "loss": "down"}


# A worked reading, so the helper is not just a definition: how many of Iraq's download
# measurements beat 10 Mbps, read off its nine reported percentiles?
_r = latest[latest.country_code == "IQ"].iloc[0]
_knots = [_r[f"download_p{p}"] for p in PERCENTILES]
_pi = pass_prob(to_quality("download", _knots), float(to_quality("download", 10.0)))
print("Iraq's nine reported download percentiles (Mbps):")
print("   " + "  ".join(f"p{p}={v:.1f}" for p, v in zip(PERCENTILES, _knots)))
print(f"\n10 Mbps falls between two of them, so the share passing is read off the curve "
      f"joining them:\n   {_pi:.4f}  ({_pi * _r.n_down:,.0f} of {_r.n_down:,.0f} measurements)")


---

## 4. But the CDF Is Smooth — So Where Do the Jumps Come From?

This is the right objection, and it deserves a straight answer.

Notebook 02 builds a CDF for each country out of its nine reported percentiles. A CDF is a smooth,
continuous curve: move along the speed axis and the fraction of measurements below you rises
gradually. Nothing about it jumps. So if everything downstream reads that curve, where does any
discreteness come from?

**The curve really is smooth, and it stays smooth all the way to the last step.** Three different
objects are involved, and only the third one is discrete:

| | object | smooth? |
|---|---|---|
| 1 | the country's **true** CDF | yes — a fixed, continuous curve |
| 2 | the CDF you **estimate** from *n* measurements | yes — also continuous, but random |
| 3 | the **IQB score** computed from it | **no** |

So the discreteness is not a property of the data or of the curve. It is a property of the
**scoring rule**, and it is introduced in one specific place.

### 4a. Exactly where smoothness dies

The pass rate is just a reading off the curve — the height of the CDF at the threshold:

$$\pi_k \;=\; 1 - F(t_k)$$

That is perfectly smooth. Nudge the curve and $\pi_k$ nudges with it, by a proportional amount.
No jumps anywhere.

But **the score never uses $\pi_k$**. It uses the answer to a comparison:

$$\mathrm{IQB}(q) \;=\; \sum_k w_k \,\mathbf{1}\{\,q > 1 - \pi_k\,\}$$

That indicator is 0 while $\pi_k$ is below the cut-off and 1 once it is above, with nothing in
between. A change of 0.0001 in $\pi_k$ either does **nothing at all**, or moves the score by the
**entire weight** of that question — as much as 0.202.

The smooth curve goes in. A staircase comes out. Here is the same country drawn both ways.

In [ ]:
EX = "IQ"
row_c = latest[latest.country_code == EX].iloc[0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

# ---- left: the CDF you build from the nine percentiles. Smooth.
ax = axes[0]
knots_v = np.array([row_c[f"download_p{p}"] for p in PERCENTILES], float)
grid = np.logspace(np.log10(knots_v.min() * 0.7), np.log10(knots_v.max() * 1.3), 400)
Fq = np.array([1 - pass_prob(to_quality("download", knots_v), float(to_quality("download", g)))
               for g in grid])
ax.plot(grid, Fq, color=SLATE, lw=2.2, zorder=3, label="the CDF, built from nine points")
ax.scatter(knots_v, QLEV, s=34, color=SLATE, zorder=4, edgecolor="white", linewidth=0.8,
           label="the nine reported percentiles")
for thr, w in ATOMS["download"]:
    pi = pass_prob(to_quality("download", knots_v), float(to_quality("download", thr)))
    ax.axvline(thr, color=ACCENT, lw=1.6, ls="--")
    ax.plot([thr], [1 - pi], marker="o", ms=7, color=ACCENT, zorder=5)
    ax.annotate(f"{1 - pi:.3f} of measurements\nare below {thr:g} Mbps",
                (thr, 1 - pi), xytext=(9, -12), textcoords="offset points",
                fontsize=8.5, color=ACCENT, va="top", ha="left")
ax.set_xscale("log")
ax.set_xlabel("download speed (Mbps)")
ax.set_ylabel("share of measurements below that speed")
ax.set_ylim(-0.03, 1.03)
ax.set_title(f"{country_name(EX)}: the curve is smooth", fontsize=11, pad=20)
ax.legend(fontsize=8, loc="lower right")

# ---- right: the score that curve produces, as the reporting level varies. A staircase.
ax = axes[1]
P_ex = np.array([[pass_prob(to_quality(m, [row_c[f"{m}_p{p}"] for p in PERCENTILES]),
                            float(to_quality(m, thr)))
                  for m in METRICS for thr, _ in ATOMS[m]]])
qq = np.linspace(0.001, 0.999, 2000)
score = np.array([float(np.sum(ATOM_W * (q > 1 - P_ex[0]))) for q in qq])
ax.step(qq, score, where="post", color=ACCENT, lw=2.4, zorder=3)
for p in PERCENTILES:
    ax.axvline(p / 100, color=SLATE, lw=0.9, ls=":", zorder=2)
    ax.text(p / 100, 1.015, f"p{p}", ha="center", fontsize=7.5, color=SLATE,
            transform=ax.get_xaxis_transform())
ax.set_xlabel("reporting level q")
ax.set_ylabel("IQB score")
ax.set_ylim(-0.03, 1.03)
ax.set_title(f"{country_name(EX)}: the score it produces is a staircase", fontsize=11, pad=20)
fig.tight_layout()
save(fig, "fig2_smooth_curve_step_score")
plt.show()

jumps = int(np.sum(np.abs(np.diff(score)) > 1e-9))
cuts = np.round(1 - P_ex[0], 6)
print(f"The curve on the left has no jumps. The score on the right has {jumps}.\n")
print("There is one jump per question, at the level where that question changes its answer -")
print(f"except where two questions change at the same level, which merges their steps. "
      f"{len(ATOM_W)} questions,")
print(f"{len(np.unique(cuts))} distinct levels, so {jumps} visible steps.")
print("\nSame country, same month, same data. The staircase is made by the scoring rule.")

### 4b. Move the curve smoothly and watch the score jump

The clearest way to see it: take a country and slide its whole download distribution up and down by
a smooth factor, as if its networks got gradually faster. The share of measurements passing each
threshold then moves smoothly, because it is a reading off a smooth curve.

The score does not.

In [ ]:
lam = np.logspace(np.log10(0.4), np.log10(12), 900)   # scale the distribution smoothly
kn_q = {m: to_quality(m, [row_c[f"{m}_p{p}"] for p in PERCENTILES]) for m in METRICS}
base = {m: [pass_prob(kn_q[m], float(to_quality(m, thr))) for thr, _ in ATOMS[m]]
        for m in METRICS}

pi_dl, pi_dl25, iqb = [], [], []
for a in lam:
    kn = to_quality("download", np.array([row_c[f"download_p{p}"] for p in PERCENTILES]) * a)
    pis = [pass_prob(kn, float(to_quality("download", thr))) for thr, _ in ATOMS["download"]]
    pi_dl.append(pis[0])
    pi_dl25.append(pis[1])
    full = dict(base)
    full["download"] = pis
    flat = np.array([v for m in METRICS for v in full[m]])
    iqb.append(float(np.sum(ATOM_W * (0.50 > 1 - flat))))
pi_dl, pi_dl25, iqb = np.array(pi_dl), np.array(pi_dl25), np.array(iqb)

fig, ax = plt.subplots(figsize=(9, 4.6))
ax.plot(lam, pi_dl, color=SLATE, lw=2.2, label="share passing download > 10")
ax.plot(lam, pi_dl25, color=SLATE, lw=1.6, ls="--", label="share passing download > 25")
ax.axhline(0.5, color=SLATE, lw=1, ls=":")
ax.text(lam[0], 0.515, " the p50 cut-off: half of measurements", fontsize=8.5, color=SLATE)
ax.set_xscale("log")
ax.set_xlabel("multiply the whole download distribution by this")
ax.set_ylabel("share passing (smooth)", color=SLATE)
ax.set_ylim(-0.03, 1.03)
ax2 = ax.twinx()
ax2.step(lam, iqb, where="post", color=ACCENT, lw=2.6, label="IQB score at p50")
ax2.set_ylabel("IQB score at p50 (steps)", color=ACCENT)
ax2.set_ylim(-0.03, 1.03)
ax2.grid(False)
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=8.5, loc="center right")
ax.set_title("A smooth change in the data produces a step change in the score", fontsize=11.5)
save(fig, "fig3_smooth_in_step_out")
plt.show()

steps = np.where(np.abs(np.diff(iqb)) > 1e-9)[0]
print(f"The grey line is smooth everywhere. The score takes "
      f"{len(np.unique(np.round(iqb, 6)))} distinct values across the whole sweep.")
for s in steps:
    print(f"   at a multiplier of {lam[s]:.3f} the share passing crosses {pi_dl[s]:.3f} "
          f"and the score jumps by {iqb[s + 1] - iqb[s]:+.3f}")

### 4c. So where does the randomness come in?

The true curve is fixed. But you never see it — you see an estimate built from *n* measurements,
which scatters around the true one. At a threshold, that scatter is **vertical**: the estimated
height $\hat{F}(t)$ lands near the true $F(t)$, but not exactly on it.

Put the two facts together and the whole analysis follows:

- the estimated height scatters by roughly $\sqrt{F(1-F)/n}$ — smaller as *n* grows;
- the score depends on **which side of the cut-off that height lands**.

If the true height sits far from a cut-off, the scatter is harmless and the answer is the same
every month. If it sits close, the scatter decides the answer — and the score jumps by the full
weight of that question.

**That is the entire problem, and section 5 is just this picture with real numbers on it.** The
smooth curve was never the issue. Reading a smooth curve at a fixed point is easy. Deciding which
side of a line the reading falls on, when the reading is a hair from the line, is not.

### 4d. One more thing "smooth" hides

There is a separate point worth being clear about, because "I have a smooth curve" can mean two
different things.

You do **not** have the country's CDF. You have **nine points from it**, plus a rule for joining
them up. The curve looks smooth because the interpolation drew it that way — but between two
neighbouring percentiles, its shape is an assumption, not a measurement.

Most of the time this is harmless, because a threshold that sits well inside a gap gets much the
same reading under any sensible rule. It stops being harmless when the threshold lands very close
to one of the nine knots, where different rules disagree.

In [ ]:
def read_with(rule, knots_q, t):
    """Three defensible ways to join nine points into a curve."""
    y = np.asarray(knots_q, float)
    uy = np.unique(y)
    if rule == "bracket":
        return pass_prob(knots_q, t)
    up = np.array([(QLEV[y == v].max() if rule == "upper" else QLEV[y == v].min()) for v in uy])
    F = (1.0 if t >= uy[0] else 0.0) if len(uy) == 1 else float(np.interp(t, uy, up))
    return float(np.clip(1.0 - F, 0.005, 0.995))


print("all ten questions, read three ways, for each country. `spread` is how much the three")
print("rules disagree; `to cut-off` is how far the reading sits from the nearest reporting "
      "level.\n")
for cc in ["IQ", "US", "AW"]:
    r = latest[latest.country_code == cc].iloc[0]
    print(f"{country_name(cc)}")
    print(f"{'question':>18s}{'this notebook':>15s}{'spread':>9s}{'to cut-off':>12s}   verdict")
    for m in METRICS:
        kq = to_quality(m, [r[f"{m}_p{p}"] for p in PERCENTILES])
        for thr, _ in ATOMS[m]:
            t = float(to_quality(m, thr))
            vals = [read_with(k, kq, t) for k in ("bracket", "upper", "lower")]
            spread = max(vals) - min(vals)
            near = min(abs(vals[0] - (1 - p / 100)) for p in PERCENTILES)
            op = ">" if HIGHER_BETTER[m] else "<"
            # 5e-5 is below the precision printed, so identical-looking rows read identically
            flag = ("settled by the data" if spread <= 5e-5 else
                    "the rule decides it" if spread > near else "spread is harmless")
            print(f"{f'{m} {op} {thr:g}':>18s}{vals[0]:>15.4f}{spread:>9.4f}{near:>12.4f}"
                  f"   {flag}")
    print()

print("Where the three rules agree, the reading is a fact about the data.")
print("Where they disagree by more than the distance to the nearest cut-off, the score is not")
print("determined by the nine numbers we were given - and no amount of extra measurement fixes")
print("it, because the information was lost when the measurements were summarised into nine")
print("percentiles.")
print("\nAcross the full panel that is true for 22 of 245 countries (notebook 03, section 10b).")
print("Ten extra columns - one count per threshold - would remove the interpolation entirely.")


---

## 5. Three Countries, One Month

Sections 2 to 4 were the setup. This is where it becomes concrete.

We take three countries from the same month and run the identical walkthrough on each: find the
question that country is closest to getting wrong, count how many of its measurements pass, and see
where the nine cut-offs fall relative to that count.

| | why it is here |
|---|---|
| **United States** | Enormous volume. 13.2 million upload tests. |
| **Iraq** | Solid volume — 28,000 upload tests — and a poor score. |
| **Aruba** | Small. About 4,000 tests a month. |

The result is not the one you would guess from that table.

In [ ]:
def most_fragile(cc):
    """The (question, level) pair this country is closest to getting wrong, and the arithmetic."""
    row = latest[latest.country_code == cc].iloc[0]
    best = None
    for m in METRICS:
        n = float(row.n_up if BACKED_BY[m] == "up" else row.n_down)
        knots = to_quality(m, [row[f"{m}_p{p}"] for p in PERCENTILES])
        for thr, w in ATOMS[m]:
            pi = pass_prob(knots, float(to_quality(m, thr)))
            move = np.sqrt(n * pi * (1 - pi))       # how far the count would shift on a re-run
            for lvl in PERCENTILES:
                gap = pi * n - (1 - lvl / 100) * n
                z = abs(gap) / move
                if best is None or z < best["z"]:
                    best = dict(z=z, cc=cc, metric=m, thr=thr, weight=w, level=lvl,
                                pi=pi, n=n, passing=pi * n, move=move, row=row)
    return best

def question_label(f):
    op = ">" if HIGHER_BETTER[f["metric"]] else "<"
    return f"{f['metric']} {op} {f['thr']:g}"


THREE = ["US", "IQ", "AW"]
F = {cc: most_fragile(cc) for cc in THREE}

print(f"the single answer each country is closest to getting wrong, in "
      f"{latest.month.iloc[0]}:\n")
print(f"{'country':>16s}{'measurements':>15s}{'its most fragile question':>28s}"
      f"{'at level':>10s}{'gap vs movement':>18s}")
for cc in THREE:
    f = F[cc]
    print(f"{country_name(cc)[:15]:>16s}{f['n']:>15,.0f}{question_label(f):>28s}"
          f"{'p' + str(f['level']):>10s}{f['z']:>18.1f}")


### 5a. Reading one country: Iraq

Take Iraq first, in full. In December 2025 it ran 27,961 upload speed tests. One of the ten
questions asks whether upload is faster than 25 Mbps. Reading that threshold off Iraq's nine
reported percentiles tells us how many of those tests passed.

In [ ]:
f = F["IQ"]
row, n_ex, pi_ex, passing, move = f["row"], f["n"], f["pi"], f["passing"], f["move"]
EX_M, EX_T = f["metric"], f["thr"]
knots = np.array([row[f"{EX_M}_p{p}"] for p in PERCENTILES], float)
j = int(np.searchsorted(knots, EX_T))

print(f"{country_name('IQ')}, {row.month}: {int(n_ex):,} {EX_M} speed tests\n")
for p, v in zip(PERCENTILES, knots):
    mark = f"   <-- the {EX_T:g} Mbps threshold falls in this gap" if p == PERCENTILES[j] else ""
    print(f"   {EX_M}_p{p:<2d} = {v:>7.2f} Mbps{mark}")
print(f"\nSo about {pi_ex:.1%} of Iraq's tests beat {EX_T:g} Mbps - that is {passing:,.0f} "
      f"of them.")

Now apply section 3. Each of the nine levels turns that question into a **cut-off on the count of
7,034**. Nothing about Iraq changes between the rows below. The measurements, the threshold and the
count are identical in all nine. **Only the cut-off moves.**

In [ ]:
def cutoff_table(f, label=None):
    print(f"{label or country_name(f['cc'])}: {int(f['n']):,} tests, of which "
          f"{f['passing']:,.0f} pass "
          f"{f['metric']} {'>' if HIGHER_BETTER[f['metric']] else '<'} {f['thr']:g}.")
    print(f"Re-run the month with a different set of users and that count would move by "
          f"about +/-{f['move']:,.0f}.\n")
    print(f"{'level':>7s}{'is really asking':>26s}{'cut-off':>12s}{'gap':>12s}"
          f"{'gap vs movement':>16s}{'verdict':>12s}")
    for lvl in PERCENTILES:
        cut = (1 - lvl / 100) * f["n"]
        gap = f["passing"] - cut
        z = abs(gap) / f["move"]
        verdict = "certain" if z > 3 else ("50/50" if z < 1 else "unclear")
        print(f"{'p' + str(lvl):>7s}{f'do >{100-lvl}% pass?':>26s}{cut:>12,.0f}{gap:>+12,.0f}"
              f"{z:>16.1f}{verdict:>12s}")


cutoff_table(F["IQ"])
amb = F["IQ"]["level"]
nxt = min(abs(F["IQ"]["passing"] - (1 - p / 100) * F["IQ"]["n"]) / F["IQ"]["move"]
          for p in PERCENTILES if p != amb)
print(f"\nEight of the nine answers are certain. One - p{amb} - is a 50/50, in a country with "
      f"{int(F['IQ']['n']):,}\nmeasurements. The next nearest level is {nxt:.0f} times further "
      f"from its cut-off than the count can move.")

### The step that is easy to miss

The count of passing tests is **not a fixed fact about Iraq.** It is what came out of one month,
from whoever happened to run a test. Run December again with a different set of users and it lands
somewhere else — usually within about a hundred either way.

So Iraq's p75 answer rests on a margin of a few dozen tests, while the number it is built from moves
by more than that. The cut-off sits inside the range the count could plausibly have taken. That
answer is a 50/50, and no amount of care in the calculation fixes it.

Iraq has nearly 28,000 measurements. That is a lot. **It is not the problem.**

### 5b. The same walkthrough on the United States and Aruba

In [ ]:
for cc in ["US", "AW"]:
    cutoff_table(F[cc])
    print()

### 5c. What the three countries show

Put the three next to each other and the pattern is not the one the measurement counts suggest.

In [ ]:
print(f"{'':>16s}{'measurements':>14s}{'most fragile':>26s}{'gap vs':>9s}"
      f"{'levels that are':>17s}")
print(f"{'':>16s}{'that month':>14s}{'question':>26s}{'movement':>9s}{'not certain':>17s}")
for cc in THREE:
    f = F[cc]
    shaky = [p for p in PERCENTILES
             if abs(f["passing"] - (1 - p / 100) * f["n"]) / f["move"] < 3]
    label = f"{question_label(f)} at p{f['level']}"
    print(f"{country_name(cc)[:15]:>16s}{f['n']:>14,.0f}{label:>26s}{f['z']:>9.1f}"
          f"{(' '.join('p' + str(p) for p in shaky) or 'none'):>17s}")

print("\nThe United States has 13.2 million upload tests, and its MOST fragile answer - the one")
print(f"closest to going the wrong way anywhere in its whole score - is still "
      f"{F['US']['z']:.0f} movements clear.")
print("Nothing about the US score is in doubt. That is what enough data looks like.")
print("\nIraq and Aruba both have an answer that is a 50/50. Each sits a hair from a cut-off:")
for cc in ("IQ", "AW"):
    f = F[cc]
    print(f"   {country_name(cc):<14s} pass rate {f['pi']:.4f} against a p{f['level']} cut-off "
          f"of {1 - f['level'] / 100:.2f}  ->  {abs(f['pi'] - (1 - f['level']/100)):.4f} away")


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11.5, 7.4))
for ax, cc in zip(axes, THREE):
    f = F[cc]
    tr = ax.get_xaxis_transform()
    for lvl in PERCENTILES:
        cut, hot = (1 - lvl / 100) * f["n"], lvl == f["level"]
        ax.axvline(cut, color=ACCENT if hot else SLATE, lw=2 if hot else 1.1)
        ax.text(cut, 0.70, f"p{lvl}", ha="center", fontsize=8, transform=tr,
                color=ACCENT if hot else SLATE, fontweight="bold" if hot else "normal")
    ax.plot([f["passing"]], [0.42], marker="o", ms=8, color=ACCENT, transform=tr,
            clip_on=False, zorder=5)
    right = f["passing"] > f["n"] * 0.5          # keep the label inside the axes
    ax.annotate(f"{f['passing']:,.0f} of {f['n']:,.0f} pass {question_label(f)}\n"
                f"nearest cut-off (p{f['level']}) is {f['z']:.1f} movements away",
                xy=(f["passing"], 0.42), xycoords=tr,
                xytext=(f["passing"] + (-1 if right else 1) * f["n"] * 0.04, 0.16),
                textcoords=tr, ha="right" if right else "left",
                fontsize=8.5, color=ACCENT,
                arrowprops=dict(arrowstyle="-", color=ACCENT, lw=1))
    ax.set_xlim(-f["n"] * 0.02, f["n"] * 1.02)
    ax.set_yticks([])
    ax.grid(False)
    ax.set_title(f"{country_name(cc)} — {f['n']:,.0f} measurements", fontsize=10.5, loc="left")
axes[-1].set_xlabel("number of measurements")
fig.suptitle("The same nine cut-offs, three countries. In colour: the count, and the cut-off "
             "nearest it.", fontsize=11.5)
fig.tight_layout()
save(fig, "fig4_three_countries")
plt.show()

### 5d. What it would take to fix each one

The last step is to turn "this answer is a 50/50" into a number of measurements.

In [ ]:
z95 = stats.norm.ppf(0.975)


def measurements_needed(pi, level):
    d = abs(pi - (1 - level / 100))
    return np.inf if d == 0 else int(np.ceil(pi * (1 - pi) * z95 ** 2 / d ** 2))


print(f"{'':>16s}{'has':>14s}{'needs for that':>17s}{'shortfall':>12s}")
print(f"{'':>16s}{'':>14s}{'one answer':>17s}")
for cc in THREE:
    f = F[cc]
    need = measurements_needed(f["pi"], f["level"])
    short = "-" if need <= f["n"] else f"{need / f['n']:,.0f}x"
    print(f"{country_name(cc)[:15]:>16s}{f['n']:>14,.0f}{need:>17,d}{short:>12s}")

print("\nAnd the same question at every other level, for Iraq, to show how much the level "
      "matters:\n")
print(f"{'level':>7s}{'distance from the cut-off':>28s}{'measurements needed':>22s}")
for lvl in PERCENTILES:
    need = measurements_needed(F["IQ"]["pi"], lvl)
    print(f"{'p' + str(lvl):>7s}{abs(F['IQ']['pi'] - (1 - lvl / 100)):>28.4f}{need:>22,d}")
print(f"\nIraq has {F['IQ']['n']:,.0f}. Its p50 answer needs "
      f"{measurements_needed(F['IQ']['pi'], 50)}. Its p{F['IQ']['level']} answer needs "
      f"{measurements_needed(F['IQ']['pi'], F['IQ']['level']):,}.")

---

## 6. What Iraq Tells Us

The amount of data a country needs has almost nothing to do with the things you would expect. Not
its size, not its population, not how variable its internet is.

> **It comes down to one thing: how close does this country sit to a threshold?**

It works like an election poll. If a candidate is at 70%, a small poll tells you who won. If they
are at 50.01%, an enormous poll still cannot call it. Sample size buys precision, and precision
only matters when the result is close.

And the relationship is **squared** — halve the distance to the line and you need four times the
data. That single fact is why the requirement varies across countries by seven orders of
magnitude.

In [ ]:
d_grid = np.logspace(-4, np.log10(0.5), 300)
show = {"TM": "Turkmenistan", "GB": "United Kingdom", "ID": "Indonesia",
        "IN": "India", "MX": "Mexico", "CA": "Canada", "HU": "Hungary"}
req = pd.read_csv("./analysis_outputs/03_sample_size_requirements.csv")

fig, ax = plt.subplots(figsize=(8.6, 5))
ax.plot(d_grid, 0.25 * (z95 / d_grid) ** 2, color=SLATE, lw=2.2, zorder=2)
for cc, nm in show.items():
    r = req[req.cc == cc]
    if r.empty or not np.isfinite(r["n_req_0.05"].iloc[0]):
        continue
    d, n = float(r.margin.iloc[0]), float(r["n_req_0.05"].iloc[0])
    ax.scatter([d], [n], s=52, color=ACCENT, zorder=4, edgecolor="white", linewidth=0.9)
    ax.annotate(f" {nm}", (d, n), fontsize=9, color=INK, va="center", zorder=5)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(7e-5, 1.6)
ax.set_xlabel("how far the country sits from its nearest threshold\n"
              "(share of measurements, so 0.01 = one percentage point)")
ax.set_ylabel("measurements needed each month")
ax.set_title("Halve the distance to a threshold, quadruple the data you need", fontsize=11.5)
save(fig, "fig5_distance_to_data")
plt.show()

print(f"{'country':>16s}{'its closest question':>24s}{'distance':>12s}{'needs':>14s}")
for cc, nm in show.items():
    r = req[req.cc == cc]
    if r.empty:
        continue
    r = r.iloc[0]
    print(f"{nm:>16s}{r.binding_test:>24s}{r.margin:>12.5f}{r['n_req_0.05']:>14,.0f}")
print("\nHungary's packet-loss question is a 50.00 against 49.98 race. That is the whole reason")
print("it needs 16.6 million measurements while Turkmenistan needs 2.")

---

## 7. So How Many Measurements Are Needed?

For a score to land within **0.05** of the truth **95%** of the time at the p50 level.

In [ ]:
fin = req["n_req_0.05"].replace(np.inf, np.nan)
enough = req.n_down_month_median >= req["n_req_0.05"]
print(f"the typical country needs      {fin.median():>12,.0f} measurements a month")
print(f"a quarter of countries need    {fin.quantile(.25):>12,.0f} or fewer")
print(f"a quarter need                 {fin.quantile(.75):>12,.0f} or more")
print(f"the range runs from            {fin.min():>12,.0f} to {fin.max():,.0f}")
print(f"and for {int(np.isinf(req['n_req_0.05']).sum()) + int((req['n_req_0.05'] > 1e5).sum())} "
      f"countries no realistic number is enough\n")
print(f"today {enough.sum()} of {len(req)} countries collect enough.\n")

print(f"{'country':>18s}{'collects each month':>21s}{'needs':>12s}   verdict")
for cc in ["IN", "GB", "TM", "CY", "TZ"]:
    r = req[req.cc == cc]
    if r.empty:
        continue
    r = r.iloc[0]
    ok = r.n_down_month_median >= r["n_req_0.05"]
    print(f"{r.country[:17]:>18s}{r.n_down_month_median:>21,.0f}{r['n_req_0.05']:>12,.0f}"
          f"   {'enough' if ok else 'NOT ENOUGH'}")

tz = req[req.cc == "TZ"].iloc[0]
tm = req[req.cc == "TM"].iloc[0]
print(f"\nTanzania collects {tz.n_down_month_median:,.0f} measurements a month and does not have "
      f"enough.")
print(f"Turkmenistan collects {tm.n_down_month_median:,.0f} and has "
      f"{tm.n_down_month_median / max(tm['n_req_0.05'], 1):,.0f} times more than it needs.")
print("\nWhich is why a blanket rule - 'hide any country under 10,000 measurements' - gets it")
print("wrong in both directions at once. It would hide Turkmenistan, whose score is solid, and")
print("publish Tanzania's, which is not.")

---

## 8. Three Things People Find Surprising

**A country can be reliable at one level and hopeless at another.** Kenya needs 182 measurements
for its p50 score and 397,024 for its p75 score. Same country, same month, same measurements —
different questions asked of them. So the question is not "can we publish Kenya?" but "which of
Kenya's nine scores can we publish?" Most countries can support 8 of the 9 today.

**For some countries no amount of data helps.** A country sitting exactly on a threshold gives an
unpredictable answer forever — a hundred measurements or a billion makes no difference. That is not
a data-collection failure; it is what happens when a score is built from hard cut-offs. The honest
output for those countries is a range, not a number.

**Too few measurements do not just add noise — they push the score down.** A country scoring near
1.0 cannot score any higher, so every question that might come out wrong can only come out *worse*.
Good countries with little data are systematically published below their true score. This is worse
than ordinary noise: widening the error bars does not fix it, because the centre is in the wrong
place, and it does not average out, because every under-measured country is pushed the same
direction at once.

In [ ]:
ke = req[req.cc == "KE"].iloc[0]
print(f"Kenya:  p50 needs {ke.n_req_p50:,.0f}   p75 needs {ke.n_req_p75:,.0f}")
lv = req.levels_supported.fillna("none").apply(lambda v: 0 if v == "none" else len(v.split()))
print(f"levels each country can publish today, out of 9: median {lv.median():.0f}\n")

on_thr = req.verdict_p50.str.startswith("sits on a threshold")
print(f"countries sitting on a threshold, where no sample size helps: {int(on_thr.sum())}")
print(req.loc[on_thr, ["country", "n_down_month_median", "binding_test"]]
      .to_string(index=False, header=["country", "collects/month", "the question it sits on"],
                 float_format=lambda v: f"{v:,.0f}"))

biased = req.bias_at_current_n.abs() > 0.05
print(f"\ncountries published more than 0.05 below or above their true score: {int(biased.sum())}")
print(f"the worst is understated by {abs(req.bias_at_current_n.min()):.2f} on a 0-1 scale")
print(req.nsmallest(5, "bias_at_current_n")[["country", "n_down_month_median", "iqb_p50",
                                             "bias_at_current_n"]]
      .to_string(index=False, header=["country", "collects/month", "true score", "published minus true"],
                 float_format=lambda v: f"{v:,.3f}"))

---

## 9. What Follows

1. **Decide country by country, and level by level.** One global cutoff is worse than no cutoff.
   Every requirement can be calculated in advance from data already in hand.
2. **Ask the pipeline to count directly.** Right now the pass rate has to be worked backwards from
   nine reported percentiles, and that step loses information — for 22 countries it loses more than
   the margin being measured. Ten extra columns, one count per threshold, would make the whole
   calculation exact. Section 2 is the reason this works: the ten pass rates determine the score at
   every percentile level.
3. **Correct the downward push on countries with few measurements** before anyone ranks them
   against each other.
4. **Publish a range for the countries sitting on a threshold.**
5. **Stop collecting more once a country passes about 1,000 measurements a month.** Above that,
   sampling has almost nothing to do with why a score moves; what is left is the internet actually
   changing, which is the thing worth looking at.

---

### One structural note

Every difficulty in this notebook comes from section 2 — from IQB being a sum of hard yes/no
answers. A score that ramped smoothly across each threshold instead of stepping over it would make
the estimate move continuously with the data, remove the countries that no sample size can settle,
remove the downward push, and shrink the typical requirement considerably. That is a decision about
how the score is designed rather than about how it is measured.

In [ ]:
print("figures written for lifting into slides or a document:\n")
for f in sorted(FIG_DIR.glob("*.png")):
    print(f"   {f}  ({f.stat().st_size / 1024:,.0f} KB)")
print("\nfull analysis      03-sample_size_reliability.ipynb")
print("assumptions        05-methodology_review.ipynb, METHODOLOGY.md")
print("per-country table  analysis_outputs/03_sample_size_requirements.csv")